In [30]:
import requests
from bs4 import BeautifulSoup
import re


headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;"
        "q=0.9,image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate",
    "Connection": "keep-alive",
}


def extract_performer_id(url):
    """
    Extract IBDB performer ID from URL.
    Example:
    /broadway-cast-staff/alfred-drake-4031 -> 4031
    """
    
    return url.split("-")[-1]


def extract_cast(url):
    """
    Extract Opening Night Cast from an IBDB production page.
    Returns a list of performers.
    """

    # Request page
    response = requests.get(
        url,
        headers=headers
    )

    response.raise_for_status()

    # Parse HTML
    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # Locate Opening Night Cast section
    cast_section = soup.find(
        id="OpeningNightCast"
    )

    if cast_section is None:
        raise ValueError(
            "Opening Night Cast section not found"
        )

    # Find individual cast rows
    cast_rows = cast_section.find_all(
        "div",
        class_="row mobile-a-align"
    )

    cast = []

    for row in cast_rows:

        performer = row.find(
            "a",
            href=re.compile(
                "/broadway-cast-staff/"
            )
        )

        # Ignore rows without performer links
        if performer is None:
            continue

        columns = row.find_all(
            "div",
            class_="col m4 s12"
        )

        character = None

        if len(columns) > 1:
            character = columns[1].get_text(
                strip=True
            )

        cast.append(
            {
                "performer_id": extract_performer_id(
                    performer["href"]
                ),
                "performer_name": performer.text.strip(),
                "character": character
            }
        )

    return cast

In [33]:
oklahoma_url = (
    "https://www.ibdb.com/broadway-production/oklahoma-1285"
)

cast = extract_cast(oklahoma_url)

cast[:5]

[{'performer_id': '4031',
  'performer_name': 'Alfred Drake',
  'character': 'Curly'},
 {'performer_id': '57977',
  'performer_name': 'Joan Roberts',
  'character': 'Laurey'},
 {'performer_id': '14313',
  'performer_name': 'Joseph Buloff',
  'character': 'Ali Hakim'},
 {'performer_id': '67218',
  'performer_name': 'Howard Da Silva',
  'character': 'Jud Fry'},
 {'performer_id': '38152',
  'performer_name': 'Lee Dixon',
  'character': 'Will Parker'}]

In [ ]:
pd.DataFrame(cast).head()

,performer_id,performer_name,character
0,4031,Alfred Drake,Curly
1,57977,Joan Roberts,Laurey
2,14313,Joseph Buloff,Ali Hakim
3,67218,Howard Da Silva,Jud Fry
4,38152,Lee Dixon,Will Parker


In [ ]:
len(cast)

35

IBDB production pages contain multiple categories of associated people. Searching all /broadway-cast-staff/ links returns producers, directors, designers, etc. The #OpeningNightCast anchor provides a reliable way to isolate performers belonging to the production cast.